# Clustering Pipeline — Full Evaluation

End-to-end patch-embedding clustering with every diagnostic and
statistical test from `clustering/pipeline.py` and `clustering/viz.py`.

**Pipeline:**
1. Configuration
2. Load checkpoint & build encoder
3. Extract patch embeddings
4. PCA whitening & singular-value spectrum
5. UMAP dimensionality reduction (15-D for clustering, 2-D for plots)
6. Leiden clustering + bootstrap stability
7. GMM-BIC cross-check
8. HDBSCAN fallback
9. Collapse diagnostics (effective rank, mean pairwise cosine)
10. Permutation null ARI
11. Image-level cross-validation (Caicedo 2017)
12. Cluster characterisation — medoid grids
13. Per-image cluster frequencies & chi² independence
14. Cluster purity by source image
15. Treatment-group analysis — group × cluster chi² / permutation chi²
16. MMD permutation test between groups
17. PERMANOVA on per-image frequency vectors
18. Group-level visualisations (heatmaps, boxplots, UMAP)
19. Save results & write cluster labels to index.csv

---
## 1 — Imports & setup

In [ ]:
%matplotlib inline

import os, sys, json, time, logging
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt

In [ ]:
NB_DIR = Path.cwd().resolve()
ROOT   = NB_DIR
# Walk up to find root/ (the package root)
while ROOT.name != 'root' and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('ROOT:', ROOT)

In [ ]:
from training.config import ModelCfg
from training.seeding import seed_everything
from training.logging import setup_logger
from training.augment import ValSingleViewTransform
from training.data import TransformedSubset
from models import build_swin_encoder, count_params
from utils_data.patch_dataset import PatchDataset

from clustering.core import (
    reduce_2d,
)
from clustering.pipeline import (
    ClusterCfg,
    extract_patch_embeddings,
    auto_pca_dim, tvn_centre, l2_then_pca_whiten,
    fit_umap, fit_umap_2d,
    leiden_sweep, bootstrap_stability, pick_resolution,
    gmm_bic_scan,
    run_hdbscan_leaf,
    cluster_size_stats, permutation_null_ari,
    train_on_imageset_predict_other,
    cluster_medoids,
    per_image_cluster_frequencies, chi2_independence,
    cluster_purity_by_image,
    group_cluster_test, mmd_permutation_test, permanova_frequencies,
    write_cluster_columns,
)
from clustering.viz import (
    plot_singular_value_spectrum,
    plot_resolution_stability,
    plot_bic_curve,
    plot_2d_clusters,
    plot_cluster_grid,
    plot_image_cluster_heatmap,
    plot_image_level_umap,
    plot_intra_image_entropy,
    plot_2d_groups,
    plot_group_cluster_heatmap,
    plot_group_frequency_boxplots,
)

print('All imports OK')

---
## 2 — Configuration

In [ ]:
from types import SimpleNamespace


# ── Treatment-group mapping ────────────────────────────────────────────────────
# Patterns matched against source_image filename (upper-cased).
# More specific patterns come first so e.g. PSYHARMIN is not
# misclassified as PSY.
GROUP_PATTERNS: list[tuple[str, str]] = [
    ('PSYHARMIN',    'Psilocybin + harmine'),
    ('NORPSIHARMIN', 'Norpsilocin + harmine'),
    ('BAEOHARMIN',   'Baeocystin + harmine'),
    ('NORPSI',       'Norpsilocin'),
    ('BAEO',         'Baeocystin'),
    ('PSY',          'Psilocybin'),
    ('PSI',          'Psilocin'),
    ('KONTROLA',     'Control'),
]


def classify_image(name: str) -> str | None:
    """Return the treatment group for a source_image filename, or None."""
    upper = name.upper()
    for pattern, group in GROUP_PATTERNS:
        if pattern in upper:
            return group
    return None


def build_group_map(source_images) -> dict[str, str]:
    """Auto-build group_map from source_image filenames."""
    gm = {}
    for img in set(source_images):
        g = classify_image(str(img))
        if g is not None:
            gm[str(img)] = g
    return gm


cfg_nb = SimpleNamespace(
    # --- Paths ---
    encoder_ckpt      = '../outputs/REPLACE_ME/best_model.pt',
    data_root         = '../../data/patches_128_new2',
    output_dir        = '../outputs/clustering_full',
    embedding_cache   = '../outputs/clustering_full/embeddings.npy',

    # --- Data ---
    exclude_patterns  = [],
    val_split         = 0.15,
    batch_size        = 64,
    num_workers       = 0,

    # group_map is built automatically after embeddings are extracted
    # (see section 6). Override here if you need manual control:
    group_map = None,

    seed = 42,
)

In [ ]:
from clustering.pipeline import ClusterCfg

cluster_cfg = ClusterCfg(
    seed=cfg_nb.seed,
    embedding_cache=cfg_nb.embedding_cache,
)

---
## 3 — Output directory, logger, seed & device

In [ ]:
RUN_TS = datetime.now().strftime('%Y%m%d_%H%M%S')
save_dir = Path(cfg_nb.output_dir) / f'run_{RUN_TS}'
save_dir.mkdir(parents=True, exist_ok=True)

logger = setup_logger('clustering', save_dir / 'run.log')
logger.info(f'encoder checkpoint = {cfg_nb.encoder_ckpt}')
logger.info(f'data root          = {cfg_nb.data_root}')
logger.info(f'save dir           = {save_dir}')

_ = seed_everything(cfg_nb.seed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f'device = {device}')

---
## 4 — Load checkpoint & build encoder

In [ ]:
ckpt_path = Path(cfg_nb.encoder_ckpt)
if not ckpt_path.exists():
    raise FileNotFoundError(f'checkpoint not found: {ckpt_path}')

ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
logger.info(f'ckpt keys = {sorted(ckpt.keys())}')

model_cfg = ModelCfg()
encoder = build_swin_encoder(model_cfg).to(device)
encoder.load_state_dict(ckpt['encoder_state_dict'])
encoder.eval()
logger.info(f'encoder params = {count_params(encoder) / 1e6:.2f} M')

---
## 5 — Build dataset & dataloader

In [ ]:
raw_ds = PatchDataset(
    root=cfg_nb.data_root,
    exclude_patterns=cfg_nb.exclude_patterns,
)
n_val   = int(len(raw_ds) * cfg_nb.val_split)
n_train = len(raw_ds) - n_val
g = torch.Generator().manual_seed(cfg_nb.seed)
train_subset, val_subset = random_split(raw_ds, [n_train, n_val], generator=g)

# Channel statistics from checkpoint (if available)
ch_mean = torch.tensor(ckpt.get('channel_mean', [0.0, 0.0, 0.0]))
ch_std  = torch.tensor(ckpt.get('channel_std',  [1.0, 1.0, 1.0]))

val_transform = ValSingleViewTransform(mean=ch_mean, std=ch_std)
val_ds = TransformedSubset(val_subset, transform=val_transform)

logger.info(f'Dataset: {len(raw_ds)} total, {n_train} train, {n_val} val')

---
## 6 — Extract patch embeddings

Globally-pooled deepest-stage features from the frozen encoder.
Cached to disk so re-runs are instant.

In [ ]:
Z, filenames, source_images, image_indices = extract_patch_embeddings(
    encoder, val_ds,
    device=device,
    batch_size=cfg_nb.batch_size,
    num_workers=cfg_nb.num_workers,
    cache_path=cluster_cfg.embedding_cache,
)
logger.info(f'Embeddings: N={Z.shape[0]}, D={Z.shape[1]}')
logger.info(f'Unique images: {len(set(source_images))}')

In [ ]:
# Singular values of centered embeddings (used by the spectrum plot below).
Z_t = torch.from_numpy(Z)
_Zc = (Z_t - Z_t.mean(dim=0, keepdim=True)).float()
S = torch.linalg.svdvals(_Zc).cpu().numpy()


---
## 8 — L2 normalisation + PCA whitening

Wang & Isola (2020): L2-normalisation before PCA removes scale effects.
Whitening decorrelates dimensions so UMAP/Leiden distances are meaningful.

In [ ]:
# Optional TVN (typical variation normalisation)
Z_proc = Z.copy()
if cluster_cfg.tvn_reference_pattern:
    Z_proc = tvn_centre(Z_proc, source_images, cluster_cfg.tvn_reference_pattern)
    logger.info(f'TVN applied with pattern: {cluster_cfg.tvn_reference_pattern}')

# Auto-select PCA dimensionality
pca_dim = cluster_cfg.pca_dim
if pca_dim is None:
    pca_dim = auto_pca_dim(
        Z_proc,
        target_var=cluster_cfg.pca_dim_target_var,
        hard_cap=cluster_cfg.pca_dim_max,
    )
logger.info(f'PCA dim = {pca_dim}')

P, pca_model, cum_var = l2_then_pca_whiten(Z_proc, n_components=pca_dim, seed=cluster_cfg.seed)
logger.info(f'PCA whitened: shape={P.shape}, cumulative var={cum_var:.3f}')
print(f'PCA dim: {pca_dim},  cumulative variance: {cum_var:.3f}')

In [ ]:
# Singular-value spectrum
fig, ax = plt.subplots(figsize=(8, 4))
plot_singular_value_spectrum(S, ax=ax, mark_target_var=cluster_cfg.pca_dim_target_var)
fig.savefig(save_dir / 'singular_value_spectrum.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

---
## 9 — UMAP dimensionality reduction

Two passes:
- **UMAP-15D** for clustering geometry (Chari & Pachter 2023)
- **UMAP-2D** for visualisation (larger `min_dist` avoids misleading density)

In [ ]:
U = fit_umap(P, cfg=cluster_cfg)
logger.info(f'UMAP-{cluster_cfg.umap_n_components}D: shape={U.shape}')

U2 = fit_umap_2d(P, cfg=cluster_cfg)
logger.info(f'UMAP-2D (vis): shape={U2.shape}')
print(f'UMAP-{cluster_cfg.umap_n_components}D done, UMAP-2D done')

---
## 10 — Leiden clustering + bootstrap stability

Resolution sweep with 70% bootstrap resampling → 1-NN propagation → ARI
vs full partition (Hennig 2007, Lange et al. 2004).
Pick the resolution with the highest mean ARI; tie-break by fewer clusters.

In [ ]:
summary, full = bootstrap_stability(
    U,
    resolutions=cluster_cfg.leiden_resolutions,
    k=cluster_cfg.leiden_k,
    B=cluster_cfg.bootstrap_b,
    frac=cluster_cfg.bootstrap_frac,
    seed=cluster_cfg.seed,
)

best_res, best_ari, best_std, best_k = pick_resolution(summary, full)
labels_leiden = full[best_res]

logger.info(f'Best resolution: {best_res}  ARI={best_ari:.3f}±{best_std:.3f}  K={best_k}')
print(f'Best resolution: {best_res}')
print(f'  ARI = {best_ari:.3f} ± {best_std:.3f}')
print(f'  K   = {best_k} clusters')

for r in sorted(summary.keys()):
    m, s = summary[r]
    k = len(set(full[r]))
    marker = ' <<<' if r == best_res else ''
    print(f'  res={r:.2f}  ARI={m:.3f}±{s:.3f}  K={k}{marker}')

In [ ]:
# Resolution stability plot
fig, ax = plt.subplots(figsize=(9, 4))
plot_resolution_stability(summary, full, ax=ax)
fig.savefig(save_dir / 'resolution_stability.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

In [ ]:
# 2D scatter coloured by Leiden cluster
fig, ax = plt.subplots(figsize=(8, 7))
plot_2d_clusters(
    U2, labels_leiden, ax=ax,
    title=f'UMAP-2D — Leiden (res={best_res}, K={best_k})',
)
fig.savefig(save_dir / 'umap2d_leiden.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

---
## 11 — GMM-BIC cross-check

Gaussian Mixture Model BIC scan to cross-validate the number of clusters
(McConville et al. 2019).

In [ ]:
bics = gmm_bic_scan(U, k_min=cluster_cfg.gmm_k_min, k_max=cluster_cfg.gmm_k_max,
                     seed=cluster_cfg.seed)
bic_best_k = bics[0][0]  # sorted by BIC, lowest first
logger.info(f'GMM-BIC best K = {bic_best_k}')
print(f'GMM-BIC best K = {bic_best_k}  (Leiden K = {best_k})')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
plot_bic_curve(bics, ax=ax)
fig.savefig(save_dir / 'gmm_bic.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

---
## 12 — HDBSCAN fallback

Density-based clustering as a sanity check (McInnes et al. 2017).
Useful when Leiden and GMM-BIC disagree.

In [ ]:
labels_hdbscan = run_hdbscan_leaf(
    U,
    min_cluster_size=cluster_cfg.hdb_min_cluster_size,
    min_samples=cluster_cfg.hdb_min_samples,
    method=cluster_cfg.hdb_method,
)
stats_hdb = cluster_size_stats(labels_hdbscan)
logger.info(f'HDBSCAN: {stats_hdb}')
print(f'HDBSCAN K = {stats_hdb["k"]},  noise = {stats_hdb["noise_frac"]:.1%}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
plot_2d_clusters(
    U2, labels_hdbscan, ax=ax,
    title=f'UMAP-2D — HDBSCAN (K={stats_hdb["k"]}, noise={stats_hdb["noise_frac"]:.1%})',
)
fig.savefig(save_dir / 'umap2d_hdbscan.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

---
## 13 — Cluster size statistics

Gini coefficient, max-cluster fraction, singleton fraction.
A high Gini means cluster sizes are unbalanced.

In [ ]:
# Use Leiden labels as the primary clustering
labels = labels_leiden
stats = cluster_size_stats(labels)

logger.info(f'Cluster size stats: {stats}')
for k, v in stats.items():
    print(f'  {k}: {v}')

---
## 14 — Permutation null ARI

Shuffle PCs independently and re-run UMAP+Leiden to estimate the
ARI expected under the null (Witten & Tibshirani 2010).
Real ARI should exceed this distribution.

In [ ]:
null_mean, null_std, null_max = permutation_null_ari(
    P, labels, cfg=cluster_cfg, resolution=best_res,
)
logger.info(f'Null ARI: {null_mean:.3f}±{null_std:.3f} (max={null_max:.3f})')
print(f'Null ARI:  mean={null_mean:.3f} ± {null_std:.3f},  max={null_max:.3f}')
print(f'Real ARI:  {best_ari:.3f}')
print(f'Real >> null? {"YES" if best_ari > null_max else "NO"}')

---
## 15 — Image-level cross-validation

Cluster half the images, predict the other half with 1-NN, measure ARI.
Low ARI means clusters do not generalise across biological samples
(Caicedo et al. 2017).

In [ ]:
cv_ari, n_a, n_b = train_on_imageset_predict_other(
    P, source_images, cfg=cluster_cfg, resolution=best_res,
)
logger.info(f'Image-level CV ARI = {cv_ari:.3f}  (n_a={n_a}, n_b={n_b})')
print(f'Image-level CV ARI: {cv_ari:.3f}  (train={n_a} patches, test={n_b} patches)')

---
## 16 — Cluster characterisation — medoid grids

Per-cluster medoid + random samples (Caron et al. 2018 DeepCluster).

In [ ]:
medoids = cluster_medoids(
    P, labels,
    samples_per_cluster=cluster_cfg.samples_per_cluster,
    seed=cluster_cfg.seed,
)
logger.info(f'Medoids computed for {len(medoids)} clusters')

# Show medoid grid using the raw (un-transformed) dataset
fig = plot_cluster_grid(val_ds, medoids, channel=0)
fig.savefig(save_dir / 'cluster_medoids.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

---
## 17 — Per-image cluster frequencies & chi² independence

Build (images × clusters) frequency matrix and test whether
images have significantly different cluster compositions.

In [ ]:
freq, image_names, cluster_ids, counts = per_image_cluster_frequencies(
    labels, source_images,
)
chi2_val, chi2_p, chi2_dof, n_dropped = chi2_independence(
    counts, min_image_patches=cluster_cfg.chi2_min_image_patches,
)
logger.info(f'chi² = {chi2_val:.2f}, p = {chi2_p:.4g}, dof = {chi2_dof}, dropped = {n_dropped}')
print(f'chi² independence test:')
print(f'  chi² = {chi2_val:.2f},  p = {chi2_p:.4g},  dof = {chi2_dof}')
print(f'  images dropped (< {cluster_cfg.chi2_min_image_patches} patches): {n_dropped}')

In [ ]:
# Per-image cluster frequency heatmap
fig, ax = plt.subplots(figsize=(max(8, 0.4 * len(cluster_ids) + 2),
                                max(5, 0.2 * len(image_names) + 1)))
plot_image_cluster_heatmap(freq, image_names, cluster_ids, ax=ax)
fig.savefig(save_dir / 'image_cluster_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

In [ ]:
# Image-level UMAP of frequency vectors
fig, ax = plt.subplots(figsize=(8, 7))
plot_image_level_umap(freq, image_names, seed=cluster_cfg.seed, ax=ax)
fig.savefig(save_dir / 'image_level_umap.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

---
## 18 — Cluster purity by source image

Per-cluster normalised entropy of the source-image distribution.
1.0 = perfectly mixed across images, 0.0 = dominated by one image.
Clusters dominated by one image indicate batch effects (Caicedo et al. 2017).

In [ ]:
purity_rows = cluster_purity_by_image(labels, source_images)
for r in purity_rows:
    logger.info(f'  cluster {r["cluster"]}: size={r["size"]}, '
                f'n_images={r["n_images_present"]}, '
                f'norm_entropy={r["norm_entropy"]:.3f}')
    print(f'  c{r["cluster"]:>3d}  size={r["size"]:>5d}  '
          f'images={r["n_images_present"]:>3d}  '
          f'entropy={r["norm_entropy"]:.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_intra_image_entropy(purity_rows, ax=ax)
fig.savefig(save_dir / 'cluster_purity.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

---
## 19 — Treatment-group statistical analysis

All tests below require a populated `group_map` in the configuration.
They compare cluster-composition profiles between biological groups
(e.g. psilocybin vs harmine vs control).

**Tests:**
- **Group × cluster chi²** (or permutation chi² with image-level shuffling)
- **MMD permutation test** on per-image mean embeddings (Gretton et al. 2012)
- **PERMANOVA** on per-image frequency vectors (Anderson 2001)

In [ ]:
# Auto-build group_map from source_image filenames
if cfg_nb.group_map is None:
    cfg_nb.group_map = build_group_map(source_images)

has_groups = len(cfg_nb.group_map) > 0

if has_groups:
    groups_found = sorted(set(cfg_nb.group_map.values()))
    logger.info(f'Group map: {len(cfg_nb.group_map)} images mapped to {len(groups_found)} groups')
    print(f'Groups found: {groups_found}')
    for g in groups_found:
        n = sum(1 for v in cfg_nb.group_map.values() if v == g)
        print(f'  {g}: {n} images')
else:
    print('⚠ group_map is empty — skipping treatment-group analysis.')
    print('  No filename matched any GROUP_PATTERNS entry.')

### 19a — Group × cluster chi² test

In [ ]:
if has_groups:
    gtest = group_cluster_test(
        counts, image_names, cluster_ids, cfg_nb.group_map,
        seed=cluster_cfg.seed,
    )
    logger.info(f'Group test: method={gtest.method_used}, '
                f'chi²={gtest.statistic:.2f}, p={gtest.p_value:.4g}')
    logger.info(f'  groups: {gtest.group_names}')
    logger.info(f'  n_images_per_group: {gtest.n_images_per_group}')
    print(f'Method: {gtest.method_used}')
    print(f'chi² = {gtest.statistic:.2f},  p = {gtest.p_value:.4g}')
    print(f'Groups: {gtest.group_names}')
    print(f'Images per group: {gtest.n_images_per_group}')

In [ ]:
if has_groups:
    fig, ax = plt.subplots(figsize=(max(6, 0.4 * len(cluster_ids) + 2),
                                    max(3, 0.5 * len(gtest.group_names) + 1)))
    plot_group_cluster_heatmap(
        gtest.group_counts, gtest.group_names, gtest.cluster_ids, ax=ax,
    )
    fig.savefig(save_dir / 'group_cluster_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)

### 19b — MMD permutation test

Tests whether the embedding distributions differ between groups using
per-image mean embeddings. Bonferroni-corrected for multiple comparisons.

In [ ]:
if has_groups:
    mmd_results = mmd_permutation_test(
        Z, source_images, cfg_nb.group_map,
        gamma=cluster_cfg.mmd_gamma,
        n_permutations=cluster_cfg.mmd_n_permutations,
        max_patches=cluster_cfg.mmd_max_patches,
        seed=cluster_cfg.seed,
    )
    for r in mmd_results:
        logger.info(f'MMD {r.group_pair}: MMD²={r.mmd_squared:.4f}, '
                    f'p={r.p_value:.4g}, n={r.n_per_group}')
        print(f'{r.group_pair[0]} vs {r.group_pair[1]}:  '
              f'MMD²={r.mmd_squared:.4f},  p={r.p_value:.4g}  '
              f'(n={r.n_per_group})')

### 19c — PERMANOVA on per-image frequency vectors

Non-parametric multivariate test on Bray-Curtis distances between
per-image cluster-frequency vectors (Anderson 2001).

In [ ]:
if has_groups:
    perm_res = permanova_frequencies(
        freq, image_names, cfg_nb.group_map,
        metric=cluster_cfg.permanova_metric,
        n_permutations=cluster_cfg.permanova_n_permutations,
        seed=cluster_cfg.seed,
    )
    logger.info(f'PERMANOVA: F={perm_res.f_statistic:.3f}, '
                f'p={perm_res.p_value:.4g}, R²={perm_res.r_squared:.3f}')
    print(f'PERMANOVA:')
    print(f'  F = {perm_res.f_statistic:.3f}')
    print(f'  p = {perm_res.p_value:.4g}')
    print(f'  R² = {perm_res.r_squared:.3f}')
    print(f'  n per group: {perm_res.n_per_group}')

---
## 20 — Group-level visualisations

In [ ]:
if has_groups:
    # UMAP-2D coloured by group
    fig, ax = plt.subplots(figsize=(8, 7))
    plot_2d_groups(
        U2, source_images, cfg_nb.group_map, ax=ax,
        title='UMAP-2D coloured by treatment group',
    )
    fig.savefig(save_dir / 'umap2d_groups.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)

In [ ]:
if has_groups:
    # Per-cluster frequency boxplots by group
    fig = plot_group_frequency_boxplots(
        freq, image_names, cfg_nb.group_map, cluster_ids,
    )
    fig.savefig(save_dir / 'group_frequency_boxplots.png', dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)

---
## 21 — Save results & write cluster labels to index.csv

In [ ]:
# Save embeddings, labels, frequencies
np.save(save_dir / 'embeddings.npy', Z)
np.save(save_dir / 'labels_leiden.npy', labels_leiden)
np.save(save_dir / 'labels_hdbscan.npy', labels_hdbscan)
np.save(save_dir / 'umap_15d.npy', U)
np.save(save_dir / 'umap_2d.npy', U2)
np.save(save_dir / 'pca_whitened.npy', P)
np.save(save_dir / 'frequencies.npy', freq)
np.save(save_dir / 'image_names.npy', image_names)
np.save(save_dir / 'cluster_ids.npy', cluster_ids)
np.save(save_dir / 'filenames.npy', filenames)
np.save(save_dir / 'source_images.npy', source_images)

logger.info(f'Saved arrays to {save_dir}')
print(f'Saved to {save_dir}')

In [ ]:
# Write cluster labels into the patch-directory index.csv
write_summary = write_cluster_columns(
    cfg_nb.data_root, filenames,
    column_to_labels={
        'leiden_cluster': labels_leiden,
        'hdbscan_cluster': labels_hdbscan,
    },
)
logger.info(f'index.csv update: {write_summary}')
for col, info in write_summary.items():
    print(f'  {col}: {info}')

---
## 22 — Run summary

In [ ]:
summary = {
    'pca_dim': pca_dim,
    'pca_cum_var': cum_var,
    'leiden_best_res': best_res,
    'leiden_ari': best_ari,
    'leiden_k': best_k,
    'gmm_bic_best_k': bic_best_k,
    'hdbscan_k': stats_hdb['k'],
    'hdbscan_noise_frac': stats_hdb['noise_frac'],
    'cluster_gini': stats['gini'],
    'null_ari_mean': null_mean,
    'null_ari_max': null_max,
    'image_cv_ari': cv_ari,
    'chi2_images': chi2_val,
    'chi2_images_p': chi2_p,
}

with open(save_dir / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

logger.info('=== RUN SUMMARY ===')
for k, v in summary.items():
    val = f'{v:.4f}' if isinstance(v, float) else str(v)
    logger.info(f'  {k}: {val}')
    print(f'  {k}: {val}')